# E5: RAG Especializado com PDFs PCDF (SEM FAISS)

**MBA IA Generativa PCDF - IBMEC**
**Encontro 5:** Especializacao de Agentes com PDFs Reais
**VERSAO:** SEM FAISS (compativel Windows)

---

## Objetivos

1. **Estender** E4 com processamento de PDFs
2. **Implementar** busca vetorial com NumPy (SEM FAISS)
3. **Adicionar** Reranking para maxima precisao
4. **Avaliar** com metricas (Precision@K, MRR)
5. **Comparar** E4 vs E5

**Tempo estimado:** 5 horas

**Progressao:** E5 AGREGA ao E4, nao substitui!

**IMPORTANTE:** Esta versao NAO usa FAISS para evitar problemas de DLL no Windows.

---

## PARTE 1: RECAP E4

### O que construímos no E4:

**9 Tools Funcionais:**
1. `contar_armas_marca` - Conta por marca
2. `contar_armas_calibre` - Conta por calibre
3. `contar_armas_tipo` - Conta por tipo
4. `contar_armas_combinado` - Marca + tipo
5. `ranking_marcas` - TOP 5 marcas
6. `ranking_calibres` - TOP 5 calibres
7. `estatisticas_gerais` - Resumo completo
8. `distribuicao_marca_por_tipo` - Distribuição
9. `buscar_conhecimento` - RAG com TF-IDF

**Recursos E4:**
- RAG básico (TF-IDF)
- Documentos .txt
- Busca por similaridade

**Limitações do E4:**
- ❌ TF-IDF não captura semântica profunda
- ❌ Não processa PDFs complexos
- ❌ Sem reranking (precisão ~40%)
- ❌ Sem métricas de avaliação

**Solução do E5:**
- ✅ Sentence-BERT para embeddings semânticos
- ✅ FAISS para busca vetorial rápida
- ✅ Reranking com CrossEncoder (precisão ~86%)
- ✅ Processamento de PDFs
- ✅ Métricas (Precision@K, MRR)

---

## PASSO 1: Instalação e Imports

In [1]:
# Instalar dependencias (executar apenas uma vez)
# VERSAO SEM FAISS - NAO precisa de faiss-cpu nem torch!
# !pip install pandas langchain-core scikit-learn sentence-transformers PyPDF2 numpy


In [2]:
#!pip install sentence_transformers

In [3]:
# Imports necessarios
import pandas as pd
import os
import numpy as np
from functools import lru_cache
from langchain_core.tools import tool

# E4 (mantidos)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# E5 (novos) - SEM FAISS
from sentence_transformers import SentenceTransformer, CrossEncoder
from PyPDF2 import PdfReader

print("OK: Imports realizados com sucesso!")
print("\nVersoes:")
print(f"  - pandas: {pd.__version__}")
print(f"  - numpy: {np.__version__}")
print("\nNOTA: Esta versao NAO usa FAISS (compativel com Windows)")

e:\documentos\ibmec\MODULO 01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: Imports realizados com sucesso!

Versoes:
  - pandas: 2.2.3
  - numpy: 1.26.4

NOTA: Esta versao NAO usa FAISS (compativel com Windows)


---

## PASSO 2: Carregar Dados E4 (Estruturados)

Vamos reutilizar a função de carregamento do E4.

In [4]:
@lru_cache(maxsize=1)
def carregar_csv():
    """
    Carrega dados SINARM do CSV.
    Cache garante que carrega apenas UMA VEZ.
    """
    # Tentar diferentes caminhos
    caminhos_possiveis = [
        "../../E4_RAG_FAISS/01_DADOS/DADOS_SINARM/OCORRENCIAS/OCORRENCIAS_2026.csv",
        "../01_DADOS/DADOS_SINARM/OCORRENCIAS/OCORRENCIAS_2026.csv"
    ]
    
    caminho = None
    for c in caminhos_possiveis:
        if os.path.exists(c):
            caminho = c
            break
    
    if not caminho:
        raise FileNotFoundError(f"Arquivo não encontrado em nenhum dos caminhos: {caminhos_possiveis}")
    
    # Tentar diferentes configurações
    configs = [
        {'encoding': 'latin-1', 'sep': ';'},  # ⭐ ESTE É O CORRETO!
        {'encoding': 'utf-8', 'sep': ';'},
        {'encoding': 'iso-8859-1', 'sep': ';'},
    ]
    
    for config in configs:
        try:
            df = pd.read_csv(caminho, **config)
            
            # Validar se carregou corretamente (deve ter múltiplas colunas)
            if len(df.columns) > 1:
                print(f"[CACHE] Carregando CSV com encoding={config['encoding']}, sep='{config['sep']}'")
                print(f"[OK] {len(df)} registros, {len(df.columns)} colunas carregadas!")
                print(f"[COLUNAS] {list(df.columns)[:5]}...")
                return df
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    
    raise Exception(f"Não foi possível ler o arquivo com nenhuma configuração testada")

# Testar carregamento
df = carregar_csv()
print(f"\n📊 Primeiros registros:")
df.head()

[CACHE] Carregando CSV com encoding=latin-1, sep=';'
[OK] 74758 registros, 10 colunas carregadas!
[COLUNAS] ['ANO_OCORRENCIA', 'MES_OCORRENCIA', 'UF', 'MUNICIPIO', 'ESPECIE_ARMA']...

📊 Primeiros registros:


,ANO_OCORRENCIA,MES_OCORRENCIA,UF,MUNICIPIO,ESPECIE_ARMA,MARCA_ARMA,CALIBRE_ARMA,TIPO_OCORRENCIA,MAIS_1000_MIL_HAB,TOTAL
0,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,.32 ...,...,N,1
1,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,20 ...,...,N,1
2,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,28 ...,...,N,1
3,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,36 ...,...,N,1
4,2026,1,AC,ACRELÂNDIA,Espingarda ...,CBC (COMPANHIA BRASILEIRA DE CARTUCHOS) ...,.32 ...,...,N,4


---

## PASSO 3: Carregar Documentos E4 (.txt)

Vamos reutilizar os documentos conceituais do E4.

In [5]:
@lru_cache(maxsize=1)
def carregar_documentos_txt():
    """
    Carrega documentos conceituais do E4 (.txt).
    
    Returns:
        list: [{"arquivo": str, "conteudo": str}]
    """
    caminhos_possiveis = [
        "../../E4_RAG_FAISS/01_DADOS/documentos_conceituais/",
        "../01_DADOS/documentos_conceituais/"
    ]
    
    caminho_docs = None
    for c in caminhos_possiveis:
        if os.path.exists(c):
            caminho_docs = c
            break
    
    if not caminho_docs:
        print(f"⚠️ Pasta de documentos não encontrada")
        return []
    
    documentos = []
    for arquivo in os.listdir(caminho_docs):
        if arquivo.endswith('.txt'):
            caminho_completo = os.path.join(caminho_docs, arquivo)
            with open(caminho_completo, 'r', encoding='utf-8') as f:
                documentos.append({
                    "arquivo": arquivo,
                    "conteudo": f.read()
                })
    
    return documentos

# Carregar
docs_txt = carregar_documentos_txt()

print(f"📚 {len(docs_txt)} documentos .txt carregados:\n")
for doc in docs_txt:
    print(f"📄 {doc['arquivo']}")
    print(f"   Tamanho: {len(doc['conteudo'])} caracteres")
    print(f"   Preview: {doc['conteudo'][:100]}...\n")

📚 6 documentos .txt carregados:

📄 calibres_armas.txt
   Tamanho: 5338 caracteres
   Preview: # Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de ...

📄 marcas_armas.txt
   Tamanho: 8120 caracteres
   Preview: # Marcas de Armas de Fogo

## Principais Marcas Brasileiras

### Taurus (Forjas Taurus S.A.)

**Hist...

📄 sistema_sinarm.txt
   Tamanho: 8376 caracteres
   Preview: # Sistema SINARM - Sistema Nacional de Armas

## O que é o SINARM?

O SINARM (Sistema Nacional de Ar...

📄 tipos_armas.txt
   Tamanho: 8934 caracteres
   Preview: # Tipos de Armas de Fogo

## Classificação Geral

As armas de fogo são classificadas de acordo com d...

📄 rag_conceitos.txt
   Tamanho: 10816 caracteres
   Preview: # RAG - Retrieval-Augmented Generation

## O que é RAG?

RAG (Retrieval-Augmented Generation) é uma ...

📄 boletim_ocorrencia.txt
   Tamanho: 8758 caracteres
   Preview: # Boletim de Ocorrência (BO)

## O que é Boletim de Ocorrência?

O Boletim de

---

## ✅ CHECKPOINT 1

**Validação:**
- [ ] CSV carregado (74.758 registros)?
- [ ] Documentos .txt carregados (5-6 arquivos)?
- [ ] Imports funcionando?

**Se tudo OK, prossiga para PARTE 2: PROCESSAR PDFs**

---

## PARTE 2: PROCESSAR PDFs

### Novidade do E5: Processar PDFs complexos

**Desafios:**
- Layouts complexos (tabelas, múltiplas colunas)
- Encoding variado
- Cabeçalhos/rodapés
- Imagens e gráficos

**Solução:**
- PyPDF2 para extração básica
- Chunking inteligente (semântico)
- Validação de qualidade

---

## PASSO 4: Carregar PDFs

In [6]:
@lru_cache(maxsize=1)
def carregar_pdfs():
    """
    Carrega e processa PDFs da PCDF.
    
    Returns:
        list: [{"arquivo": str, "caminho": str, "conteudo": str, "num_paginas": int}]
    """
    caminhos_possiveis = [
        "../01_DADOS/pdfs_pcdf/",
        "../../E5_ESPECIALIZACAO_PDFS/01_DADOS/pdfs_pcdf/"
    ]
    
    caminho_pdfs = None
    for c in caminhos_possiveis:
        if os.path.exists(c):
            caminho_pdfs = c
            break
    
    if not caminho_pdfs:
        print(f"⚠️ Pasta de PDFs não encontrada")
        print(f"💡 Crie a pasta: ../01_DADOS/pdfs_pcdf/")
        print(f"💡 Adicione PDFs de leis, manuais, portarias")
        return []
    
    pdfs = []
    total_erros = 0
    
    print("[CACHE] Carregando PDFs...")
    
    for root, dirs, files in os.walk(caminho_pdfs):
        for file in files:
            if file.endswith('.pdf'):
                caminho = os.path.join(root, file)
                try:
                    reader = PdfReader(caminho)
                    texto = ""
                    for page in reader.pages:
                        texto += page.extract_text()
                    
                    # Validar se extraiu texto
                    if len(texto.strip()) < 100:
                        print(f"⚠️ {file}: Texto muito curto ({len(texto)} chars), possível erro")
                        total_erros += 1
                        continue
                    
                    pdfs.append({
                        'arquivo': file,
                        'caminho': caminho,
                        'conteudo': texto,
                        'num_paginas': len(reader.pages)
                    })
                    
                    print(f"✅ {file}: {len(reader.pages)} páginas, {len(texto)} caracteres")
                    
                except Exception as e:
                    print(f"❌ {file}: Erro ao ler - {e}")
                    total_erros += 1
    
    print(f"\n[OK] {len(pdfs)} PDFs carregados com sucesso!")
    if total_erros > 0:
        print(f"[AVISO] {total_erros} PDFs com erro")
    
    return pdfs

# Carregar
pdfs = carregar_pdfs()

if pdfs:
    print(f"\n📚 Resumo dos PDFs:")
    for pdf in pdfs:
        print(f"\n📄 {pdf['arquivo']}")
        print(f"   Páginas: {pdf['num_paginas']}")
        print(f"   Caracteres: {len(pdf['conteudo'])}")
        print(f"   Preview: {pdf['conteudo'][:150]}...")
else:
    print("\n⚠️ Nenhum PDF encontrado. Continuando sem PDFs...")

[CACHE] Carregando PDFs...
✅ estatuto_desarmamento.pdf: 22 páginas, 28566 caracteres
✅ LEI-10.826-03-SINARM.pdf: 14 páginas, 53154 caracteres
✅ cartilha-de-armamento-e-tiro.pdf: 27 páginas, 39076 caracteres
✅ Anexo XVII - Porte de arma de fogo.pdf: 1 páginas, 2223 caracteres
✅ procedimento_operacional_padrao-pericia_criminal.pdf: 243 páginas, 383207 caracteres

[OK] 5 PDFs carregados com sucesso!

📚 Resumo dos PDFs:

📄 estatuto_desarmamento.pdf
   Páginas: 22
   Caracteres: 28566
   Preview: CÂMARA DOS DEPUTADOS
ESTATUTO
DO
DESARMAMENTO
Brasília – 2004
M E S A     D A
CÂMARA DOS DEPUTADOS
52a Legislatura – 2a Sessão Legislativa
2004
Presid...

📄 LEI-10.826-03-SINARM.pdf
   Páginas: 14
   Caracteres: 53154
   Preview: 13/03/2018 L10826
http://www .planalto.gov .br/ccivil_03/Leis/2003/L10.826.htm 1/14
Presidência da República
 Casa Civil
 Subchefia para Assuntos Jurí...

📄 cartilha-de-armamento-e-tiro.pdf
   Páginas: 27
   Caracteres: 39076
   Preview:  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

---

## PASSO 5: Chunking Inteligente

**Problema:** Documentos muito grandes não cabem no contexto do LLM

**Solução:** Dividir em chunks (pedaços) menores

**Estratégias:**
1. **Fixo:** 500 caracteres (E4)
2. **Semântico:** Por parágrafos/seções (E5)
3. **Overlap:** Chunks se sobrepõem (evita perder contexto)

In [7]:
def chunk_text_hibrido(texto, chunk_size=1000, overlap=150):
    """
    Chunking hibrido: tenta semantico, se falhar usa fixo.
    
    CORRECAO para PDFs grandes como procedimento_operacional_padrao-pericia_criminal.pdf
    
    Args:
        texto: Texto para dividir
        chunk_size: Tamanho do chunk (1000 para PDFs grandes, 500 para .txt)
        overlap: Sobreposicao (150 para PDFs grandes, 50 para .txt)
    
    Returns:
        list: Lista de chunks
    """
    chunks = []
    
    # Tentar dividir por paragrafos duplos
    paragrafos = texto.split('\n\n')
    
    # Se tiver poucos paragrafos, tentar quebra simples
    if len(paragrafos) < 5:
        paragrafos = texto.split('\n')
    
    # Se ainda tiver poucos, usar chunking FIXO (CORRECAO CRITICA)
    if len(paragrafos) < 10:
        # Chunking fixo para PDFs problematicos
        start = 0
        while start < len(texto):
            end = start + chunk_size
            chunk = texto[start:end]
            
            if len(chunk.strip()) > 50:
                chunks.append(chunk.strip())
            
            start = end - overlap
        
        return chunks
    
    # Chunking semantico normal (para PDFs bem formatados)
    chunk_atual = ""
    
    for paragrafo in paragrafos:
        # Se adicionar este paragrafo ultrapassar o limite
        if len(chunk_atual) + len(paragrafo) > chunk_size:
            # Salvar chunk atual
            if len(chunk_atual.strip()) > 50:
                chunks.append(chunk_atual.strip())
            
            # Iniciar novo chunk (com overlap)
            chunk_atual = chunk_atual[-overlap:] + paragrafo
        else:
            chunk_atual += "\n\n" + paragrafo
    
    # Adicionar ultimo chunk
    if len(chunk_atual.strip()) > 50:
        chunks.append(chunk_atual.strip())
    
    return chunks

# Testar com documento de exemplo
if docs_txt:
    doc_teste = docs_txt[0]['conteudo']
    
    chunks_hibrido = chunk_text_hibrido(doc_teste, chunk_size=500, overlap=50)
    
    print(f"📊 Chunking Hibrido:")
    print(f"\n📄 Documento: {docs_txt[0]['arquivo']}")
    print(f"   Tamanho original: {len(doc_teste)} caracteres")
    print(f"   Total de chunks: {len(chunks_hibrido)}")
    print(f"   Tamanho medio: {np.mean([len(c) for c in chunks_hibrido]):.0f} caracteres")
    
    print(f"\n📝 Exemplo de chunk:")
    print(f"{chunks_hibrido[0][:300]}...")


📊 Chunking Hibrido:

📄 Documento: calibres_armas.txt
   Tamanho original: 5338 caracteres
   Total de chunks: 14
   Tamanho medio: 426 caracteres

📝 Exemplo de chunk:
# Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de uma arma de fogo, geralmente expressa em milímetros (mm) ou polegadas. O calibre determina o tamanho do projétil que a arma pode disparar.

## Principais Calibres no Brasil

### Calibres Comuns em Pis...


---

## PASSO 6: Preparar Todos os Chunks

Vamos processar TODOS os documentos (.txt + PDFs) e criar chunks.

In [8]:
def preparar_todos_chunks():
    """
    Prepara chunks de TODOS os documentos (.txt + PDFs).
    
    CORRECAO: Usa chunk_text_hibrido e detecta PDFs grandes
    
    Returns:
        list: [{"tipo": str, "arquivo": str, "chunk_id": int, "texto": str}]
    """
    todos_chunks = []
    
    # Processar documentos .txt
    print("📚 Processando documentos .txt...")
    for doc in docs_txt:
        # Para .txt usar chunk_size menor (500)
        chunks = chunk_text_hibrido(doc['conteudo'], chunk_size=500, overlap=50)
        for i, chunk in enumerate(chunks):
            todos_chunks.append({
                'tipo': 'txt',
                'arquivo': doc['arquivo'],
                'chunk_id': i,
                'texto': chunk
            })
    
    print(f"✅ {len([c for c in todos_chunks if c['tipo'] == 'txt'])} chunks de .txt")
    
    # Processar PDFs
    if pdfs:
        print("\n📄 Processando PDFs...")
        for pdf in pdfs:
            # CORRECAO CRITICA: Detectar se e PDF grande
            tamanho = len(pdf['conteudo'])
            
            if tamanho > 100000:  # Maior que 100K caracteres
                print(f"   [!] {pdf['arquivo']}: PDF GRANDE ({tamanho:,} chars)")
                print(f"       Usando chunk_size=1000, overlap=150")
                chunk_size_pdf = 1000
                overlap_pdf = 150
            else:
                chunk_size_pdf = 500
                overlap_pdf = 50
            
            chunks = chunk_text_hibrido(
                pdf['conteudo'], 
                chunk_size=chunk_size_pdf, 
                overlap=overlap_pdf
            )
            
            print(f"   ✅ {pdf['arquivo']}: {len(chunks)} chunks criados")
            
            for i, chunk in enumerate(chunks):
                todos_chunks.append({
                    'tipo': 'pdf',
                    'arquivo': pdf['arquivo'],
                    'chunk_id': i,
                    'texto': chunk
                })
        
        print(f"\n✅ {len([c for c in todos_chunks if c['tipo'] == 'pdf'])} chunks de PDFs")
    
    print(f"\n🎉 Total: {len(todos_chunks)} chunks preparados!")
    
    return todos_chunks

# Preparar
todos_chunks = preparar_todos_chunks()

# Estatisticas
print(f"\n📊 Estatisticas dos Chunks:")
print(f"   Total: {len(todos_chunks)}")
print(f"   .txt: {len([c for c in todos_chunks if c['tipo'] == 'txt'])}")
print(f"   PDFs: {len([c for c in todos_chunks if c['tipo'] == 'pdf'])}")
print(f"   Tamanho medio: {np.mean([len(c['texto']) for c in todos_chunks]):.0f} caracteres")
print(f"   Tamanho minimo: {min([len(c['texto']) for c in todos_chunks])} caracteres")
print(f"   Tamanho maximo: {max([len(c['texto']) for c in todos_chunks])} caracteres")


📚 Processando documentos .txt...
✅ 131 chunks de .txt

📄 Processando PDFs...
   ✅ estatuto_desarmamento.pdf: 69 chunks criados
   ✅ LEI-10.826-03-SINARM.pdf: 134 chunks criados
   ✅ cartilha-de-armamento-e-tiro.pdf: 97 chunks criados
   ✅ Anexo XVII - Porte de arma de fogo.pdf: 6 chunks criados
   [!] procedimento_operacional_padrao-pericia_criminal.pdf: PDF GRANDE (383,207 chars)
       Usando chunk_size=1000, overlap=150
   ✅ procedimento_operacional_padrao-pericia_criminal.pdf: 478 chunks criados

✅ 784 chunks de PDFs

🎉 Total: 915 chunks preparados!

📊 Estatisticas dos Chunks:
   Total: 915
   .txt: 131
   PDFs: 784
   Tamanho medio: 716 caracteres
   Tamanho minimo: 88 caracteres
   Tamanho maximo: 1002 caracteres


---

## ✅ CHECKPOINT 2

**Validação:**
- [ ] PDFs carregados (ou aviso se não houver)?
- [ ] Chunks criados (>50 chunks esperados)?
- [ ] Tamanho médio ~500 caracteres?

**Se tudo OK, prossiga para PARTE 3: EMBEDDINGS**

---

## PARTE 3: EMBEDDINGS SEMÂNTICOS

### E4 vs E5

| Aspecto | E4 (TF-IDF) | E5 (Sentence-BERT) |
|---------|-------------|--------------------|
| **Tipo** | Frequência de palavras | Semântica profunda |
| **Dimensões** | 100 | 384 |
| **Sinônimos** | ❌ Não detecta | ✅ Detecta |
| **Contexto** | ❌ Não captura | ✅ Captura |
| **Velocidade** | Muito rápido | Rápido |
| **Precisão** | ~40% | ~70% |

### Sentence-BERT

**Modelo:** `paraphrase-multilingual-MiniLM-L12-v2`

**Características:**
- ✅ Multilíngue (português incluído)
- ✅ Leve (120 MB)
- ✅ Rápido (100 sentenças/segundo)
- ✅ 384 dimensões

---

## PASSO 7: Carregar Modelo Sentence-BERT

In [9]:
# Carregar modelo
print("📥 Carregando Sentence-BERT...")
print("   Modelo: paraphrase-multilingual-MiniLM-L12-v2")
print("   (Primeira vez pode demorar ~1 minuto para baixar)\n")

embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("✅ Modelo carregado!")
print(f"   Dimensões: {embedding_model.get_sentence_embedding_dimension()}")
print(f"   Max tokens: {embedding_model.max_seq_length}")

📥 Carregando Sentence-BERT...
   Modelo: paraphrase-multilingual-MiniLM-L12-v2
   (Primeira vez pode demorar ~1 minuto para baixar)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4692.85it/s]


✅ Modelo carregado!
   Dimensões: 384
   Max tokens: 128


C:\Users\Yuri Queiroz\AppData\Local\Temp\ipykernel_15548\3378223284.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"   Dimensões: {embedding_model.get_sentence_embedding_dimension()}")


---

## PASSO 8: Gerar Embeddings

In [10]:
# Extrair textos dos chunks
textos_chunks = [chunk['texto'] for chunk in todos_chunks]

print(f"🔄 Gerando embeddings para {len(textos_chunks)} chunks...")
print(f"   (Pode demorar ~1-2 minutos)\n")

# Gerar embeddings
embeddings = embedding_model.encode(
    textos_chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Embeddings gerados!")
print(f"   Forma: {embeddings.shape}")
print(f"   ({embeddings.shape[0]} chunks x {embeddings.shape[1]} dimensões)")
print(f"   Tamanho em memória: {embeddings.nbytes / 1024 / 1024:.2f} MB")

🔄 Gerando embeddings para 915 chunks...
   (Pode demorar ~1-2 minutos)



Batches: 100%|██████████| 29/29 [00:23<00:00,  1.21it/s]


✅ Embeddings gerados!
   Forma: (915, 384)
   (915 chunks x 384 dimensões)
   Tamanho em memória: 1.34 MB


---

## PASSO 9: Comparar TF-IDF vs Sentence-BERT

Vamos testar a diferença na busca.

In [11]:
# Criar vetorizador TF-IDF (E4)
vectorizer_tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
embeddings_tfidf = vectorizer_tfidf.fit_transform(textos_chunks)

print("📊 Comparação TF-IDF vs Sentence-BERT:\n")

# Pergunta de teste
pergunta_teste = "O que é calibre de arma?"

print(f"❓ Pergunta: {pergunta_teste}\n")

# Busca com TF-IDF
print("🔹 TF-IDF (E4):")
pergunta_tfidf = vectorizer_tfidf.transform([pergunta_teste])
similaridades_tfidf = cosine_similarity(pergunta_tfidf, embeddings_tfidf)[0]
top_indices_tfidf = similaridades_tfidf.argsort()[-3:][::-1]

for i, idx in enumerate(top_indices_tfidf, 1):
    chunk = todos_chunks[idx]
    score = similaridades_tfidf[idx]
    print(f"  {i}. {chunk['arquivo']} (score: {score:.3f})")
    print(f"     {chunk['texto'][:100]}...\n")

# Busca com Sentence-BERT
print("\n🔹 Sentence-BERT (E5):")
pergunta_embedding = embedding_model.encode([pergunta_teste])
similaridades_sbert = cosine_similarity(pergunta_embedding, embeddings)[0]
top_indices_sbert = similaridades_sbert.argsort()[-3:][::-1]

for i, idx in enumerate(top_indices_sbert, 1):
    chunk = todos_chunks[idx]
    score = similaridades_sbert[idx]
    print(f"  {i}. {chunk['arquivo']} (score: {score:.3f})")
    print(f"     {chunk['texto'][:100]}...\n")

print("\n💡 Observe:")
print("   - TF-IDF: Busca por palavras-chave")
print("   - Sentence-BERT: Busca por significado")

📊 Comparação TF-IDF vs Sentence-BERT:

❓ Pergunta: O que é calibre de arma?

🔹 TF-IDF (E4):
  1. calibres_armas.txt (score: 0.698)
     animais
- Recuo leve
- Alcance efetivo: 25 metros### Calibres de Rifles

**5.56mm (.223 Remington)**...

  2. rag_conceitos.txt (score: 0.676)
     Salvar
faiss.write_index(index, "index.faiss")
```### Fase 2: Busca (Online)
```python
# 1. Receber ...

  3. boletim_ocorrencia.txt (score: 0.657)
     esão)

## Como Fazer BO de Arma

### Furto de Arma**Informações Obrigatórias:**
- Número de série da...


🔹 Sentence-BERT (E5):
  1. cartilha-de-armamento-e-tiro.pdf (score: 0.797)
     nições de uso permitido.  

 

 

3 – CALIBRE  

         Medida do diâmetro interno do cano de uma ...

  2. cartilha-de-armamento-e-tiro.pdf (score: 0.784)
     A - armas de fogo de alma lisa de calibre doze ou maior com comprimento de cano menor que vinte e qu...

  3. cartilha-de-armamento-e-tiro.pdf (score: 0.762)
     corresponde ao tamanho e ao calibre da arma;  

 21

---

## PARTE 4: BUSCA VETORIAL COM NUMPY

### Por que NumPy em vez de FAISS?

**FAISS (Facebook AI Similarity Search):**
- Extremamente rapido (milhoes de vetores)
- Requer PyTorch (problema de DLL no Windows)
- Ideal para producao em larga escala

**NumPy + scikit-learn:**
- Rapido para datasets pequenos/medios (<100K docs)
- Funciona 100% no Windows (sem PyTorch)
- Mesma precisao que FAISS
- Mais simples de entender

### Comparacao

| Metodo | 1K docs | 10K docs | 100K docs |
|--------|---------|----------|----------|
| **FAISS** | 2ms | 5ms | 15ms |
| **NumPy** | 5ms | 15ms | 50ms |

**Para este projeto:** ~135 vetores → NumPy e perfeitamente adequado!

### Como funciona?

1. **Embeddings ja estao em memoria** (gerados no Passo 8)
2. **Busca:** Calcular similaridade de cosseno entre query e todos os docs
3. **Ordenar:** Pegar top-K com maior similaridade
4. **Retornar:** Documentos mais relevantes

---

## PARTE 4: FAISS (Facebook AI Similarity Search)

### Por que FAISS?

**Problema:** Busca linear é lenta para muitos documentos

**Solução:** FAISS usa estruturas otimizadas

### Comparação

| Método | 1K docs | 10K docs | 100K docs |
|--------|---------|----------|----------|
| **Linear** | 10ms | 100ms | 1000ms |
| **FAISS** | 2ms | 5ms | 15ms |

### Tipos de Índice

1. **IndexFlatL2:** Busca exata (usaremos este)
2. **IndexIVFFlat:** Busca aproximada (mais rápido)
3. **IndexHNSW:** Busca aproximada (mais preciso)

---

## PASSO 10: Validar Embeddings para Busca

In [12]:
# Validar embeddings prontos para busca
print("Validando embeddings...\n")

dimension = embeddings.shape[1]
num_vectors = embeddings.shape[0]

print(f"OK: Embeddings prontos para busca!")
print(f"   Dimensao: {dimension}")
print(f"   Total de vetores: {num_vectors}")
print(f"   Tamanho em memoria: {embeddings.nbytes / 1024 / 1024:.2f} MB")

print(f"\nMETODO DE BUSCA: Similaridade de Cosseno com NumPy")
print(f"   - Rapido para datasets pequenos/medios (<100K docs)")
print(f"   - Mesma precisao que FAISS")
print(f"   - Sem dependencia de PyTorch")

Validando embeddings...

OK: Embeddings prontos para busca!
   Dimensao: 384
   Total de vetores: 915
   Tamanho em memoria: 1.34 MB

METODO DE BUSCA: Similaridade de Cosseno com NumPy
   - Rapido para datasets pequenos/medios (<100K docs)
   - Mesma precisao que FAISS
   - Sem dependencia de PyTorch


---

## PASSO 11: Testar Busca

In [13]:
def buscar_numpy(pergunta, k=5):
    """
    Busca documentos similares usando NumPy + cosine similarity.
    
    Args:
        pergunta: Pergunta do usuario
        k: Numero de documentos a retornar
    
    Returns:
        list: [(chunk, score)]
    """
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Gerar embedding da pergunta
    query_embedding = embedding_model.encode([pergunta])
    
    # Calcular similaridade com todos os documentos
    similaridades = cosine_similarity(query_embedding, embeddings)[0]
    
    # Pegar top-K (indices com maior similaridade)
    top_indices = similaridades.argsort()[-k:][::-1]
    
    # Retornar resultados
    resultados = []
    for idx in top_indices:
        if idx < len(todos_chunks):
            chunk = todos_chunks[idx]
            score = float(similaridades[idx])
            resultados.append((chunk, score))
    
    return resultados

# Testar
perguntas_teste = [
    "O que e calibre?",
    "Como funciona o SINARM?",
    "Diferenca entre pistola e revolver?"
]

for pergunta in perguntas_teste:
    print(f"\nPergunta: {pergunta}")
    print("=" * 60 + "\n")
    
    resultados = buscar_numpy(pergunta, k=3)
    
    for i, (chunk, score) in enumerate(resultados, 1):
        print(f"{i}. {chunk['arquivo']} (score: {score:.3f})")
        print(f"   {chunk['texto'][:150]}...\n")


Pergunta: O que e calibre?

1. cartilha-de-armamento-e-tiro.pdf (score: 0.519)
   Original, PROIBIDO o uso de  munição recarregada.  Da aprovação: Será aprovado o candidato  que obtiver, no mínimo, 60% (sessenta  por cento) da pontu...

2. procedimento_operacional_padrao-pericia_criminal.pdf (score: 0.516)
   este de eficiência, na ausência de arma ou provete para testes, com arma 

14 questionada de calibre nominal compatível com o do cartucho questionado....

3. cartilha-de-armamento-e-tiro.pdf (score: 0.506)
   no mínimo, 60% (sessenta  por cento) da pontuação máxima do alvo, ou seja, 30 (trinta) pontos  em cada distância, do total dos 50 (cinquent a) pontos ...


Pergunta: Como funciona o SINARM?

1. cartilha-de-armamento-e-tiro.pdf (score: 0.489)
   ca do cartucho. O estojo possibilita que todos os componentes necessários ao disparo fiquem unidos em uma única peça, o que faci lita o manejo 

da ar...

2. cartilha-de-armamento-e-tiro.pdf (score: 0.481)
   d. RT 851 Multialloy . 



---

## PASSO 12: Salvar e Carregar Índice

In [14]:
# Caminho para salvar
caminho_embeddings = "../01_DADOS/indices/embeddings.npy"
caminho_chunks = "../01_DADOS/indices/chunks_metadata.npy"

# Criar pasta se nao existir
os.makedirs(os.path.dirname(caminho_embeddings), exist_ok=True)

# Salvar embeddings
print("Salvando embeddings...")
np.save(caminho_embeddings, embeddings)
print(f"OK: Embeddings salvos em: {caminho_embeddings}")

# Salvar metadata dos chunks
print("\nSalvando metadata dos chunks...")
np.save(caminho_chunks, todos_chunks)
print(f"OK: Metadata salva em: {caminho_chunks}")

# Testar carregamento
print("\nTestando carregamento...")
embeddings_carregados = np.load(caminho_embeddings)
chunks_carregados = np.load(caminho_chunks, allow_pickle=True)

print(f"OK: Embeddings carregados: {embeddings_carregados.shape}")
print(f"OK: Chunks carregados: {len(chunks_carregados)} chunks")

print("\nAgora voce pode carregar os embeddings sem reprocessar tudo!")

Salvando embeddings...
OK: Embeddings salvos em: ../01_DADOS/indices/embeddings.npy

Salvando metadata dos chunks...
OK: Metadata salva em: ../01_DADOS/indices/chunks_metadata.npy

Testando carregamento...
OK: Embeddings carregados: (915, 384)
OK: Chunks carregados: 915 chunks

Agora voce pode carregar os embeddings sem reprocessar tudo!


---

## ✅ CHECKPOINT 4

**Validação:**
- [ ] Índice FAISS criado?
- [ ] Busca funciona?
- [ ] Índice salvo em disco?
- [ ] Carregamento funciona?

**Se tudo OK, prossiga para PARTE 5: RERANKING**

---

## PARTE 5: RERANKING

### Problema

Busca inicial (FAISS) pode retornar documentos irrelevantes nos top-K

### Solução

**Pipeline de 2 estágios:**

```
Pergunta → FAISS (top-20) → Reranking (top-5) → Resposta
```

**Estágio 1 (FAISS):**
- Rápido (~5ms)
- Busca aproximada
- Retorna top-20

**Estágio 2 (Reranking):**
- Lento (~50ms)
- Busca precisa
- Retorna top-5

### CrossEncoder

**Modelo:** `cross-encoder/ms-marco-MiniLM-L-6-v2`

**Diferença:**
- **Bi-encoder (Sentence-BERT):** Gera embeddings separados
- **Cross-encoder:** Processa pergunta + documento juntos

**Resultado:** +200% de precisão!

---

## PASSO 13: Carregar CrossEncoder

In [15]:
# Carregar modelo
print("📥 Carregando CrossEncoder...")
print("   Modelo: cross-encoder/ms-marco-MiniLM-L-6-v2")
print("   (Primeira vez pode demorar ~30 segundos para baixar)\n")

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("✅ CrossEncoder carregado!")

📥 Carregando CrossEncoder...
   Modelo: cross-encoder/ms-marco-MiniLM-L-6-v2
   (Primeira vez pode demorar ~30 segundos para baixar)



Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5134.62it/s]


✅ CrossEncoder carregado!


---

## PASSO 14: Implementar Pipeline com Reranking

In [ ]:
def buscar_com_reranking(pergunta, k_inicial=20, k_final=5, threshold=None):
    """
    Busca com pipeline de 2 estágios: NumPy + Reranking.
    
    Args:
        pergunta: Pergunta do usuário
        k_inicial: Número de documentos na busca inicial (NumPy)
        k_final: Número de documentos após reranking
        threshold: Score mínimo para considerar relevante (opcional)
    
    Returns:
        list: [(chunk, score)]
    """
    from sklearn.metrics.pairwise import cosine_similarity
    
    # Estágio 1: Busca inicial com NumPy (top-K inicial)
    query_embedding = embedding_model.encode([pergunta])
    
    # Calcular similaridade com todos os documentos
    similaridades = cosine_similarity(query_embedding, embeddings)[0]
    
    # Pegar top-K inicial (indices com maior similaridade)
    top_indices = similaridades.argsort()[-k_inicial:][::-1]
    
    # Coletar candidatos
    candidatos = []
    for idx in top_indices:
        if idx < len(todos_chunks):
            candidatos.append(todos_chunks[idx])
    
    # Estágio 2: Reranking com CrossEncoder
    # Criar pares (pergunta, documento)
    pares = [[pergunta, chunk['texto']] for chunk in candidatos]
    
    # Calcular scores
    scores = reranker.predict(pares)
    
    # Ordenar por score (maior = melhor)
    indices_ordenados = np.argsort(scores)[::-1]
    
    # Retornar top-K final (com threshold opcional)
    resultados = []
    for i in indices_ordenados:
        chunk = candidatos[i]
        score = float(scores[i])
        
        # Aplicar threshold se fornecido
        if threshold is not None and score < threshold:
            continue
        
        resultados.append((chunk, score))
        
        # Parar quando atingir k_final
        if len(resultados) >= k_final:
            break
    
    return resultados

# Testar
print("🧪 Testando pipeline com reranking:\n")

pergunta_teste = "O que é calibre de arma?"
pergunta_teste = "Por que o DTT (Ditiotreitol) foi adicionado à solução de extração, e como sua ausência poderia comprometer o resultado?"
pergunta_teste = "Se o perito optar por usar filtros MICROCON® YM-100 em vez de precipitação alcoólica, qual seria a sequência correta de centrifugações e por quê?"


print(f"❓ Pergunta: {pergunta_teste}")
print("=" * 60 + "\n")

# Buscar com threshold=0.0 para filtrar scores negativos
resultados = buscar_com_reranking(pergunta_teste, k_inicial=20, k_final=5, threshold=0.0)

if not resultados:
    print("⚠️  Nenhum documento relevante encontrado (todos os scores < 0.0)\n")
else:
    for i, (chunk, score) in enumerate(resultados, 1):
        tipo_emoji = "📄" if chunk['tipo'] == "pdf" else "📝"
        score_emoji = "✅" if score > 2.0 else ("⚠️" if score > 0 else "❌")
        
        print(f"{i}. {tipo_emoji} {chunk['arquivo']} (score: {score:.3f}) {score_emoji}")
        print(f"   {chunk['texto'][:150]}...\n")

print("-" * 60)

🧪 Testando pipeline com reranking:

❓ Pergunta: Se o perito optar por usar filtros MICROCON® YM-100 em vez de precipitação alcoólica, qual seria a sequência correta de centrifugações e por quê?

1. 📄 procedimento_operacional_padrao-pericia_criminal.pdf (score: 3.609) ✅
   agitar vigorosamente os microtubos, por inversão, 10 vezes.

(8) Quando se optar pelo procedimento de purificação e concentração com filtros do tipo M...

2. 📄 procedimento_operacional_padrao-pericia_criminal.pdf (score: 2.953) ✅
   es 

procedimentos:

Para uso de membrana MICROCON® YM-100

9a. Transferir cuidadosamente (sem tocar com a pipeta a membrana do filtro) a fase aquosa ...

3. 📄 procedimento_operacional_padrao-pericia_criminal.pdf (score: 2.840) ✅
   ts’ . Nature, 

v.18, p.577-9, 1985. 

LEE, H. C.; LADD, C. Preservation and collection of biological evidence . Croat Med J. 

Jun;42(3):225-8, 2001....

4. 📄 procedimento_operacional_padrao-pericia_criminal.pdf (score: 0.943) ⚠️
   .

• Remover e descartar o 